In [32]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

sys.path.insert(0, str(Path("../").resolve()))

from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report
from src.risk_engine import RiskEngine

In [33]:
train = pd.read_csv("../data/processed/train_processed.csv")
test = pd.read_csv("../data/processed/test_processed.csv")

X_train = train.drop("is_fraud", axis=1)
X_test = test.drop("is_fraud", axis=1)

y_train = train["is_fraud"]
y_test = test["is_fraud"]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (1296675, 15)
Testing: (555719, 15)


In [34]:
isolation_forest = IsolationForest(
    n_estimators=150,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

isolation_forest.fit(X_train)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",150
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [35]:
anomaly_predictions = isolation_forest.predict(X_test)

anomaly_predictions = np.where(
    anomaly_predictions == -1,
    1,
    0
)

print("Anomalous transactions:", anomaly_predictions.sum())

Anomalous transactions: 214467


In [36]:
print(
    classification_report(
        y_test,
        anomaly_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0       1.00      0.62      0.76    553574
           1       0.01      0.80      0.02      2145

    accuracy                           0.62    555719
   macro avg       0.50      0.71      0.39    555719
weighted avg       0.99      0.62      0.76    555719



In [37]:
anomaly_scores = -isolation_forest.score_samples(X_test)

print("Minimum anomaly score:", anomaly_scores.min())
print("Maximum anomaly score:", anomaly_scores.max())
print("Mean anomaly score:", anomaly_scores.mean())

Minimum anomaly score: 0.4201609721343947
Maximum anomaly score: 0.6817330635078728
Mean anomaly score: 0.4953585943990907


In [38]:
joblib.dump(
    isolation_forest,
    "../models/isolation_forest.pkl"
)

['../models/isolation_forest.pkl']

In [39]:
fraud_model = joblib.load("../models/fraud_model.pkl")
scaler = joblib.load("../models/scaler.pkl")
isolation_forest = joblib.load("../models/isolation_forest.pkl")

In [40]:
fraud_probabilities = fraud_model.predict_proba(X_test)[:, 1]

print("Fraud probabilities:", fraud_probabilities[:5])
print("Anomaly scores:", anomaly_scores[:5])

Fraud probabilities: [0.00061967 0.00043151 0.00043151 0.00047606 0.00056168]
Anomaly scores: [0.45548858 0.47748776 0.46857581 0.48842556 0.52576248]


In [41]:
risk_engine = RiskEngine(
    fraud_weight=0.7,
    anomaly_weight=0.3
)

In [42]:
anomaly_min = anomaly_scores.min()
anomaly_max = anomaly_scores.max()

print("Minimum:", anomaly_min)
print("Maximum:", anomaly_max)

Minimum: 0.4201609721343947
Maximum: 0.6817330635078728


In [43]:
risk_result = risk_engine.calculate_risk(
    fraud_probability=float(fraud_probabilities[0]),
    anomaly_score=float(anomaly_scores[0]),
    anomaly_min=float(anomaly_min),
    anomaly_max=float(anomaly_max)
)

risk_result

{'fraud_probability': 0.0006196667671173542,
 'anomaly_score': 0.4554885787064707,
 'normalized_anomaly_score': 0.1350587762883102,
 'risk_score': 0.04095139962347521,
 'risk_level': 'LOW'}

In [44]:
all_risk_results = []

for index in range(len(X_test)):
    result = risk_engine.calculate_risk(
        fraud_probability=float(fraud_probabilities[index]),
        anomaly_score=float(anomaly_scores[index]),
        anomaly_min=float(anomaly_min),
        anomaly_max=float(anomaly_max)
    )

    result["index"] = index
    all_risk_results.append(result)

all_risk_df = pd.DataFrame(all_risk_results)

all_risk_df.head()

,fraud_probability,anomaly_score,normalized_anomaly_score,risk_score,risk_level,index
0,0.000620,0.455489,0.135059,0.040951,LOW,0
1,0.000432,0.477488,0.219162,0.066051,LOW,1
2,0.000432,0.468576,0.185092,0.055830,LOW,2
3,0.000476,0.488426,0.260978,0.078627,LOW,3
4,0.000562,0.525762,0.403719,0.121509,LOW,4


In [45]:
all_risk_df["risk_level"].value_counts()

risk_level
LOW       553729
HIGH        1254
MEDIUM       736
Name: count, dtype: int64

In [46]:
all_risk_df.sort_values(
    "risk_score",
    ascending=False
).head(20)

,fraud_probability,anomaly_score,normalized_anomaly_score,risk_score,risk_level,index
327423,0.990777,0.668443,0.949191,0.978301,HIGH,327423
327373,1.000000,0.660569,0.919089,0.975727,HIGH,327373
428593,0.981957,0.671063,0.959208,0.975132,HIGH,428593
85700,0.966869,0.678273,0.986774,0.972840,HIGH,85700
89926,0.964122,0.678886,0.989114,0.971620,HIGH,89926
425118,0.977049,0.668921,0.951017,0.969240,HIGH,425118
454003,1.000000,0.651062,0.882742,0.964823,HIGH,454003
400011,1.000000,0.646693,0.866039,0.959812,HIGH,400011
330135,0.987667,0.653212,0.890962,0.958655,HIGH,330135
85779,0.959814,0.668723,0.950262,0.956948,HIGH,85779


In [47]:
all_risk_df["actual_fraud"] = y_test.values

pd.crosstab(
    all_risk_df["risk_level"],
    all_risk_df["actual_fraud"]
)

actual_fraud,0,1
risk_level,,
HIGH,164,1090
LOW,553097,632
MEDIUM,313,423


In [48]:
all_risk_df.to_csv(
    "../reports/risk_analysis.csv",
    index=False
)

In [49]:
print("Isolation Forest completed")
print("Risk analysis completed")
print("Risk results saved to reports/risk_analysis.csv")
print("Total transactions analyzed:", len(all_risk_df))
print("\nRisk distribution:")
print(all_risk_df["risk_level"].value_counts())

Isolation Forest completed
Risk analysis completed
Risk results saved to reports/risk_analysis.csv
Total transactions analyzed: 555719

Risk distribution:
risk_level
LOW       553729
HIGH        1254
MEDIUM       736
Name: count, dtype: int64
